## Lab6 - LZW

In [54]:
import math
import pickle
import heapq
from bitarray import bitarray
from collections import Counter
import os

In [55]:
# Huffman code

class HuffmanNode:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None

    def __lt__(self, other):
        # Dla kolejki priorytetowej
        return self.freq < other.freq

class HuffmanCompressor:
    def __init__(self):
        self.char_to_code = {}
        self.code_to_char = {}
        self.root = None

    def create(self, frequencies):
        """Tworzy drzewo Huffmana i kody na podstawie częstości występowania znaków."""
        if not frequencies:
            return

        # Tworzymy kolejkę priorytetową z liści
        priority_queue = [HuffmanNode(char, freq) for char, freq in frequencies.items()]
        heapq.heapify(priority_queue)

        # Budujemy drzewo
        while len(priority_queue) > 1:
            left = heapq.heappop(priority_queue)
            right = heapq.heappop(priority_queue)

            merged = HuffmanNode(None, left.freq + right.freq)
            merged.left = left
            merged.right = right

            heapq.heappush(priority_queue, merged)

        self.root = priority_queue[0]
        self.char_to_code = {}
        self._build_codes(self.root, "")
        self.code_to_char = {v: k for k, v in self.char_to_code.items()}
        
        print(f"Liczba unikalnych znaków: {len(frequencies)}")

    def _build_codes(self, node, current_code):
        if node is None:
            return

        # Jeśli to liść, przypisujemy kod
        if node.char is not None:
            # Specjalny przypadek: tylko jeden unikalny znak w tekście
            self.char_to_code[node.char] = current_code if current_code else "0"
            return

        self._build_codes(node.left, current_code + "0")
        self._build_codes(node.right, current_code + "1")

    def encode(self, text):
        """Zamienia tekst na bitarray."""
        bit_string = "".join(self.char_to_code[char] for char in text)
        return bitarray(bit_string)

    def decode(self, bits, original_text_len):
        """Dekoduje bity z powrotem na tekst."""
        if not self.root:
            return ""
            
        decoded_chars = []
        current_node = self.root
        bits_str = bits.to01()
        bit_idx = 0
        
        # Dekodujemy aż do uzyskania original_text_len znaków
        while len(decoded_chars) < original_text_len and bit_idx < len(bits_str):
            bit = bits_str[bit_idx]
            bit_idx += 1
            
            if bit == '0':
                current_node = current_node.left
            else:
                current_node = current_node.right
            
            if current_node.char is not None:
                decoded_chars.append(current_node.char)
                current_node = self.root
                
        return "".join(decoded_chars)

    def save(self, filename, bits, original_len):
        """Zapisuje kod i dane do pliku."""
        data = {
            'char_to_code': self.char_to_code,
            'encoded_bits': bits.tobytes(),
            'num_bits': len(bits),
            'original_len': original_len
        }
        with open(filename, 'wb') as f:
            pickle.dump(data, f)

    def load(self, filename):
        """Wczytuje kod i dane z pliku."""
        with open(filename, 'rb') as f:
            data = pickle.load(f)
        
        self.char_to_code = data['char_to_code']
        self.code_to_char = {v: k for k, v in self.char_to_code.items()}
        
        # Odtwarzamy drzewo na podstawie kodów
        self._rebuild_tree()
        
        bits = bitarray()
        bits.frombytes(data['encoded_bits'])
        # Przycinamy do faktycznej liczby bitów (padding bajtowy)
        bits = bits[:data['num_bits']]
        return bits, data['original_len']

    def _rebuild_tree(self):
        """Odtwarza drzewo Huffmana z mapowania char_to_code."""
        self.root = HuffmanNode(None, 0)
        for char, code in self.char_to_code.items():
            current_node = self.root
            for bit in code:
                if bit == '0':
                    if current_node.left is None:
                        current_node.left = HuffmanNode(None, 0)
                    current_node = current_node.left
                else:
                    if current_node.right is None:
                        current_node.right = HuffmanNode(None, 0)
                    current_node = current_node.right
            current_node.char = char

def load_text(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        return f.read()

def calculate_stats(text, compressor):
    n = len(text)
    freqs = Counter(text)
    probs = {c: f / n for c, f in freqs.items()}
    
    # Entropia H(X)
    entropy = -sum(p * math.log2(p) for p in probs.values())
    
    # Średnia długość kodu L
    avg_len = sum(probs[char] * len(compressor.char_to_code[char]) for char in freqs)
    
    # Efektywność eta
    efficiency = (entropy / avg_len) * 100 if avg_len > 0 else 0
    
    return entropy, avg_len, efficiency

## LZW - kompresja i dekompresja

In [56]:
def lzw_compress(text, max_dict_size=4096):
    if not text:
        return [], {}, {}
        
    is_bytes = isinstance(text, bytes)
    unique_chars = sorted(list(set(text)))
    
    if is_bytes:
        dict = {bytes([c]): i for i, c in enumerate(unique_chars)}
    else:
        dict = {c: i for i, c in enumerate(unique_chars)}
        
    original_dict = dict.copy()
    output = []
    
    string = text[0:1] 

    for i in range(1, len(text)):
        char = text[i:i+1] 
        
        if string + char in dict:
            string += char
        else:
            output.append(dict[string])
            if len(dict) < max_dict_size:
                dict[string + char] = len(dict)
            string = char
            
    if string:
        output.append(dict[string])
    
    return output, dict, original_dict


def lzw_decompress(compressed, dict):
    if not compressed:
        return b"" if isinstance(list(dict.keys())[0], bytes) else ""
        
    reverse_dict = {v: k for k, v in dict.items()}

    old = compressed[0]
    output = reverse_dict[old]
    
    c = output[0:1] 
    
    for i in compressed[1:]:
        if i in reverse_dict: 
            word = reverse_dict[i]
        else:
            word = reverse_dict[old] + c

        output += word
        c = word[0:1] 
        reverse_dict[len(reverse_dict)] = reverse_dict[old] + c
        old = i
        
    return output

In [57]:
def summarize_results(original_text, compressed_data, result):
    if isinstance(original_text, str):
        original_size = len(original_text.encode('utf-8')) * 8 
    else:
        original_size = len(original_text) * 8  # Bajty nie wymagają encode()

    # Obliczamy rozmiar skompresowanych danych (w bitach) w zależności od typu
    if isinstance(compressed_data, bitarray):
        compressed_size = len(compressed_data)
    elif isinstance(compressed_data, (bytes, bytearray)):
        compressed_size = len(compressed_data) * 8
    elif isinstance(compressed_data, list):
        # Jeśli to lista kodów (np. LZW), estymujemy liczbę bitów na kod
        if len(compressed_data) == 0:
            compressed_size = 0
        else:
            max_code = max(compressed_data)
            bits_per_code = math.ceil(math.log2(max_code + 1)) if max_code > 0 else 1
            compressed_size = len(compressed_data) * bits_per_code
    else:
        # Fallback: próbujemy użyć długości i traktujemy jako bajty
        try:
            compressed_size = len(compressed_data) * 8
        except Exception:
            compressed_size = 0

    compression_ratio = compressed_size / original_size if original_size > 0 else 0
    print(f"Rozmiar oryginalny: {original_size} bitów")
    print(f"Rozmiar skompresowany: {compressed_size} bitów")
    print(f"Współczynnik kompresji (skompresowany/original): {compression_ratio:.4f}")
    if original_text == result:
        print("Dekodowanie poprawne: Oryginalny tekst i zdekodowany tekst są identyczne.")
    else:
        print("Dekodowanie niepoprawne: Oryginalny tekst i zdekodowany tekst różnią się.")

### Główna metoda do kompresji i dekompresji

In [58]:
def compress_and_decompress(filename, double_compress=False):
    binary_extensions = [".bin", ".bmp", ".jpg", ".jpeg", ".png", ".pdf", ".zip", ".rar"]
    if os.path.splitext(filename)[1] in binary_extensions:
        with open(filename, 'rb') as f:
            data = f.read()
        print(f"Odczytano plik binarny: {filename} ({len(data)} bajtów)")
        is_binary = True
    else:
        with open(filename, 'r', encoding='utf-8') as f:
            data = f.read()
        print(f"Odczytano plik tekstowy: {filename} ({len(data)} znaków)")
        is_binary = False
    
    if double_compress:
        print("Wykonanaie podwójnej kompresji: LZW + Huffman")
        lzw_compressed, lzw_dict, original_dict = lzw_compress(data)
        
        # Konwersja listy intów (kody LZW) do stringa przed Huffmanem
        huffman_input = ''.join(chr(code) for code in lzw_compressed)

        # Kompresja Huffmanem na wynik LZW (już jako string)
        compressor = HuffmanCompressor()
        freqs = Counter(huffman_input)
        compressor.create(freqs)
        encoded_data = compressor.encode(huffman_input)

        # Dekodowanie Huffmana — otrzymamy string, który trzeba zamienić na listę intów
        decoded_string = compressor.decode(encoded_data, len(huffman_input))
        decoded_ints = [ord(c) for c in decoded_string]

        # Dekompresja LZW z listy intów
        lzw_decompressed = lzw_decompress(decoded_ints, original_dict)

        summarize_results(data, encoded_data, lzw_decompressed)
    else:
        print("Wykonanaie pojedynczej kompresji: LZW")
        lzw_compressed, lzw_dict, original_dict = lzw_compress(data)
        lzw_decompressed = lzw_decompress(lzw_compressed, original_dict)
        summarize_results(data, lzw_compressed, lzw_decompressed)
    
    file_name = os.path.splitext(filename)
    if double_compress:
        new_filename = f"{file_name[0]}_compressed_double{file_name[1]}"
    else:
        new_filename = f"{file_name[0]}_compressed{file_name[1]}"
    if is_binary:
        with open(new_filename, 'wb') as f:
            f.write(data)
    else:
        with open(new_filename, 'w', encoding='utf-8') as f:
            f.write(data)
    print(f"Zapisano skompresowany plik: {new_filename}")
        

### Kompresja i dekompresja poszczególnych plików

In [59]:
# 'norm_wiki_sample.txt'

# LZW + Huffman
compress_and_decompress('norm_wiki_sample.txt', double_compress=True)

print("\n" + "="*50 + "\n")

# LZW
compress_and_decompress('norm_wiki_sample.txt', double_compress=False)

Odczytano plik tekstowy: norm_wiki_sample.txt (10788941 znaków)
Wykonanaie podwójnej kompresji: LZW + Huffman
Liczba unikalnych znaków: 4063
Rozmiar oryginalny: 86311528 bitów
Rozmiar skompresowany: 39208227 bitów
Współczynnik kompresji (skompresowany/original): 0.4543
Dekodowanie poprawne: Oryginalny tekst i zdekodowany tekst są identyczne.
Zapisano skompresowany plik: norm_wiki_sample_compressed_double.txt


Odczytano plik tekstowy: norm_wiki_sample.txt (10788941 znaków)
Wykonanaie pojedynczej kompresji: LZW
Rozmiar oryginalny: 86311528 bitów
Rozmiar skompresowany: 43176996 bitów
Współczynnik kompresji (skompresowany/original): 0.5002
Dekodowanie poprawne: Oryginalny tekst i zdekodowany tekst są identyczne.
Zapisano skompresowany plik: norm_wiki_sample_compressed.txt


In [60]:
# 'wiki_sample.txt'

# LZW + Huffman
compress_and_decompress('wiki_sample.txt', double_compress=True)

print("\n" + "="*50 + "\n")

# LZW
compress_and_decompress('wiki_sample.txt', double_compress=False)

Odczytano plik tekstowy: wiki_sample.txt (11904620 znaków)
Wykonanaie podwójnej kompresji: LZW + Huffman
Liczba unikalnych znaków: 4043
Rozmiar oryginalny: 95236960 bitów
Rozmiar skompresowany: 46489899 bitów
Współczynnik kompresji (skompresowany/original): 0.4881
Dekodowanie poprawne: Oryginalny tekst i zdekodowany tekst są identyczne.
Zapisano skompresowany plik: wiki_sample_compressed_double.txt


Odczytano plik tekstowy: wiki_sample.txt (11904620 znaków)
Wykonanaie pojedynczej kompresji: LZW
Rozmiar oryginalny: 95236960 bitów
Rozmiar skompresowany: 52769016 bitów
Współczynnik kompresji (skompresowany/original): 0.5541
Dekodowanie poprawne: Oryginalny tekst i zdekodowany tekst są identyczne.
Zapisano skompresowany plik: wiki_sample_compressed.txt


In [61]:
# 'lena.bmp'

# LZW + Huffman
compress_and_decompress('lena.bmp', double_compress=True)

print("\n" + "="*50 + "\n")

# LZW
compress_and_decompress('lena.bmp', double_compress=False)

Odczytano plik binarny: lena.bmp (11524938 bajtów)
Wykonanaie podwójnej kompresji: LZW + Huffman
Liczba unikalnych znaków: 4018
Rozmiar oryginalny: 92199504 bitów
Rozmiar skompresowany: 83542002 bitów
Współczynnik kompresji (skompresowany/original): 0.9061
Dekodowanie poprawne: Oryginalny tekst i zdekodowany tekst są identyczne.
Zapisano skompresowany plik: lena_compressed_double.bmp


Odczytano plik binarny: lena.bmp (11524938 bajtów)
Wykonanaie pojedynczej kompresji: LZW
Rozmiar oryginalny: 92199504 bitów
Rozmiar skompresowany: 107882592 bitów
Współczynnik kompresji (skompresowany/original): 1.1701
Dekodowanie poprawne: Oryginalny tekst i zdekodowany tekst są identyczne.
Zapisano skompresowany plik: lena_compressed.bmp
